IMDB film yorumlarını veri seti üzerinden GRU tabanlı bir duygu analizi modeli

In [1]:
pip install tensorflow keras

In [2]:
import numpy as np
from tensorflow.keras.datasets import imdb #hazır veri seti
from tensorflow.keras.preprocessing.sequence import pad_sequences #padding
from tensorflow.keras.models import Sequential #model kurma için
from tensorflow.keras.layers import Embedding, GRU, Dense #katmanlar

In [5]:
#imdb veri setinde en sık geçen 1bin adet kelime kullanılsın
num_words=10000 #sözlükte tutulacak kelime sayısı
max_sequence_length=200 # her yorumu 200 kelime ile sabitle

# X_train ve X_test = yorumlar
# y_train ve y_test = etiketler  negatife 1 pozitif
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

print(f"Train boyutu: {len(X_train)}, test boyutu: {len(X_test)}")

Train boyutu: 25000, test boyutu: 25000


In [6]:
#padding
X_train_padded= pad_sequences(X_train, maxlen= max_sequence_length)
X_test_padded= pad_sequences(X_test, maxlen= max_sequence_length)

print(f"X_train şekli: {X_train_padded.shape}")
print(f"X_test şekli: {X_test_padded.shape}")

X_train şekli: (25000, 200)
X_test şekli: (25000, 200)


In [7]:
#model oluşturma
embedding_dim=100 # kelimeler 100 boyutlu vektörler ile temsil edilsin

model = Sequential()

#Embedding layer
# imput_dim=sözlükteki kelime sayısı
#output_dim= her kelime için vektör boyutu
#input_length= her yorumun uzunluğu
model.add(Embedding(input_dim=num_words, output_dim=embedding_dim, input_length=max_sequence_length))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
#GRU layer
#units=64, gru hücrelerinin gizli durum boyutu
#return_sequences=False: sadece son çıktıyı döndür(Birçok-to-bit problem)
model.add(GRU(units=64, return_sequences=False))

#output layer
#dense(1) ikili sınıflandırma
model.add(Dense(1, activation="sigmoid"))

In [9]:
#model derleme
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [10]:
#model training
history=model.fit(X_train_padded, y_train, epochs=3, batch_size=128, validation_split=0.2, verbose=1)

#model değerlendirme
loss,accuracy=model.evaluate(X_test_padded, y_test, verbose=1)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

Epoch 1/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 78s 481ms/step - accuracy: 0.7248 - loss: 0.5248 - val_accuracy: 0.8348 - val_loss: 0.3888
Epoch 2/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 85s 501ms/step - accuracy: 0.8824 - loss: 0.2879 - val_accuracy: 0.8540 - val_loss: 0.3402
Epoch 3/3
157/157 ━━━━━━━━━━━━━━━━━━━━ 80s 513ms/step - accuracy: 0.9217 - loss: 0.2053 - val_accuracy: 0.8536 - val_loss: 0.3374
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 35ms/step - accuracy: 0.8514 - loss: 0.3522
Test loss: 0.3522
Test accuracy: 0.8514


In [11]:
#yeni cümle tahmini
#imdb dataset sayısaş index kullandığı için doğrudan kelimeler ile test edemiyoruz bunun için word_index alıp index_to_word mapping yapmamız lazım

word_index=imdb.get_word_index() #kelime-> index sözlüğü
#indexleri kelimeye çevirmek için test mappign oluştur
index_to_word={v+3: k for k, v in word_index.items()}
index_to_word[0]="<PAD>" #padding
index_to_word[1]="<START>" #cümlenin başlangıcı
index_to_word[2]="<UNK>" #bilinmeyen cümle

def decode_review(encoded_review):
  #sayısal bir yorumu tekrar kelimelere çevirir
  return " ".join([index_to_word.get(i, "?") for i in encoded_review])


def classify_review(review_sequence):
  #sayısal imdb yorumunu sınıflandırır
  padded=pad_sequences([review_sequence], maxlen=max_sequence_length)
  prob=model.predict(padded)[0][0]
  label="positive" if prob >0.5 else "negative"
  return label,prob


decoded=decode_review(X_test[0])
print(decoded)
pred_label,prob=classify_review(X_test[0])
print(f"Tahmin: {pred_label}, olasılık: {prob}")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
<START> please give this one a miss br br <UNK> <UNK> and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite <UNK> so all you madison fans give this a miss
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step
Tahmin: negative, olasılık: 0.2911531329154968
